# 33a — Applications in NLP

Corresponds to: Note 33 — Applications in NLP

**Libraries introduced in this notebook:** No new libraries — PyTorch (`torch`, `torch.nn`) was introduced in Note 09.

**What this notebook demonstrates:**
1. Skip-gram forward pass trace — matching the Note 33 worked example exactly
2. Word embedding lookup with `nn.Embedding` — the standard PyTorch way to implement the input matrix
3. Sentiment classifier — LSTM encoder with embedding layer, trained on a small synthetic dataset
4. Perplexity computation — evaluating a language model on a held-out sequence

## Part 1 — Skip-Gram Forward Pass Trace

We reproduce the Note 33 worked example: vocabulary size 5, embedding dimension 2, centre word "deep", context word "learning".

In [1]:
import torch
import torch.nn.functional as F

# ── Vocabulary ────────────────────────────────────────────────────────────────
vocab = ['deep', 'learning', 'neural', 'network', 'model']
w2i   = {w: i for i, w in enumerate(vocab)}

# ── Input embedding for 'deep' ────────────────────────────────────────────────
v_deep = torch.tensor([0.5, -0.3], dtype=torch.float64)

# ── Output vectors (rows = words) ────────────────────────────────────────────
U = torch.tensor([
    [-0.1,  0.5],   # deep
    [ 0.4,  0.6],   # learning
    [-0.2,  0.3],   # neural
    [ 0.1, -0.4],   # network
    [ 0.3,  0.2],   # model
], dtype=torch.float64)

# ── Scores: s_w = u_w @ v_deep ────────────────────────────────────────────────
scores = U @ v_deep
print('Scores s_w:')
for w, s in zip(vocab, scores):
    print(f'  {w:10s}: {s.item():.4f}')

# ── Softmax probabilities ─────────────────────────────────────────────────────
probs = F.softmax(scores, dim=0)
print('\nSoftmax probabilities:')
for w, p in zip(vocab, probs):
    print(f'  {w:10s}: {p.item():.4f}')

# ── Loss: -log P(learning | deep) ────────────────────────────────────────────
target_idx = w2i['learning']
loss = -torch.log(probs[target_idx])
print(f'\nP(learning | deep): {probs[target_idx].item():.4f}')
print(f'Loss:               {loss.item():.4f}')

Scores s_w:
  deep      : -0.2000
  learning  : 0.0200
  neural    : -0.1900
  network   : 0.1700
  model     : 0.0900

Softmax probabilities:
  deep      : 0.1656
  learning  : 0.2063
  neural    : 0.1672
  network   : 0.2397
  model     : 0.2213

P(learning | deep): 0.2063
Loss:               1.5785


Expected from Note 33:
- `s_learning` = 0.02, `s_neural` = -0.19, `s_network` = 0.17, `s_model` = 0.09, `s_deep` = -0.20
- `P(learning | deep)` ≈ 0.2063
- Loss ≈ 1.5773

## Part 2 — Word Embedding Lookup with nn.Embedding

`nn.Embedding` is the standard PyTorch layer for the input matrix $W_{\text{in}}$. It maps integer token indices to dense vectors.

In [2]:
import torch.nn as nn
torch.manual_seed(0)

V, d = 5, 2
embedding = nn.Embedding(num_embeddings=V, embedding_dim=d)

# Look up embeddings for all words
indices = torch.arange(V)
vecs    = embedding(indices)

print('Embedding matrix (V x d):')
for w, v in zip(vocab, vecs):
    print(f'  {w:10s}: {v.detach().numpy().round(4)}')

print(f'\nEmbedding weight shape: {embedding.weight.shape}')

Embedding matrix (V x d):
  deep      : [ 1.541  -0.2934]
  learning  : [-2.1788  0.5684]
  neural    : [-1.0845 -1.3986]
  network   : [0.4033 0.838 ]
  model     : [-0.7193 -0.4033]

Embedding weight shape: torch.Size([5, 2])


## Part 3 — Sentiment Classifier

We build a simple LSTM-based sentiment classifier. The model embeds tokens, encodes the sequence with an LSTM, and classifies using the final hidden state.

In [3]:
torch.manual_seed(1)

# ── Tiny synthetic dataset ────────────────────────────────────────────────────
# Vocabulary: 0=good, 1=great, 2=bad, 3=terrible, 4=PAD
# Label: 1=positive, 0=negative
data = [
    (torch.tensor([0, 1, 4, 4]), 1),   # "good great" → positive
    (torch.tensor([2, 3, 4, 4]), 0),   # "bad terrible" → negative
    (torch.tensor([1, 1, 0, 4]), 1),   # "great great good" → positive
    (torch.tensor([3, 2, 3, 4]), 0),   # "terrible bad terrible" → negative
    (torch.tensor([0, 0, 1, 1]), 1),   # "good good great great" → positive
    (torch.tensor([2, 2, 3, 3]), 0),   # "bad bad terrible terrible" → negative
]

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, embed_dim, padding_idx=4)
        self.lstm   = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc     = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        e = self.embed(x)                    # (B, T, embed_dim)
        _, (h, _) = self.lstm(e)             # h: (1, B, hidden_dim)
        return self.fc(h.squeeze(0))         # (B, 2)

model     = SentimentLSTM(vocab_size=5, embed_dim=4, hidden_dim=8)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

for epoch in range(200):
    total_loss = 0
    for x, y in data:
        logits = model(x.unsqueeze(0))       # add batch dim
        loss   = criterion(logits, torch.tensor([y]))
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        total_loss += loss.item()

# Evaluate
correct = 0
model.eval()
with torch.no_grad():
    for x, y in data:
        pred = model(x.unsqueeze(0)).argmax(dim=1).item()
        correct += (pred == y)

print(f'Training accuracy: {correct}/{len(data)}')

Training accuracy: 6/6


## Part 4 — Perplexity

We compute perplexity for a simple bigram language model on a short sequence. Perplexity = $\exp(-\frac{1}{T}\sum_t \log P(w_t | w_{t-1}))$.

In [4]:
import math

# ── Bigram probabilities (hand-specified) ─────────────────────────────────────
# P(next | current) for a 4-word vocabulary: [deep, learning, neural, network]
# Rows = current word, Cols = next word
bigram_probs = torch.tensor([
    [0.05, 0.60, 0.25, 0.10],   # after 'deep'
    [0.10, 0.05, 0.70, 0.15],   # after 'learning'
    [0.20, 0.30, 0.05, 0.45],   # after 'neural'
    [0.50, 0.20, 0.25, 0.05],   # after 'network'
])

# Test sequence: deep → learning → neural → network
# Indices:        0         1          2         3
sequence = [0, 1, 2, 3]

log_probs = []
for t in range(1, len(sequence)):
    p = bigram_probs[sequence[t-1], sequence[t]].item()
    log_probs.append(math.log(p))
    print(f'P({vocab[sequence[t]]} | {vocab[sequence[t-1]]}) = {p:.2f}  log = {math.log(p):.4f}')

avg_nll = -sum(log_probs) / len(log_probs)
ppl     = math.exp(avg_nll)
print(f'\nAverage NLL: {avg_nll:.4f}')
print(f'Perplexity:  {ppl:.4f}')

P(learning | deep) = 0.60  log = -0.5108
P(neural | learning) = 0.70  log = -0.3567
P(network | neural) = 0.45  log = -0.7985

Average NLL: 0.5553
Perplexity:  1.7425


A lower perplexity means the model assigns higher probability to the observed sequence — it is less "surprised" by the text.

## Summary

This notebook demonstrated:
1. Skip-gram forward pass — scores, softmax, and cross-entropy loss matching the Note 33 worked example
2. `nn.Embedding` — the standard PyTorch embedding lookup layer
3. LSTM sentiment classifier — embedding + LSTM encoder + linear head, trained on a synthetic dataset
4. Perplexity — computed from bigram probabilities on a short sequence